
# 🪄 Story-SLM: Pretraining a Small Language Model From Scratch for Creative Story Generation

This notebook trains a **decoder-only Transformer (GPT-style) Small Language Model (SLM) completely from scratch** — custom tokenizer, custom architecture, custom training loop — to generate short creative stories.

It is built to run on **Kaggle** (free GPU: T4 x2 is recommended), which means:
- Sessions have a time limit (~9-12h) and can disconnect.
- So this notebook has **full checkpointing + resume support** — you can stop and re-run it across multiple Kaggle sessions and it will pick up exactly where it left off.
- It also periodically saves generated sample stories so you can watch quality improve as training goes.

### Plan
1. Setup & config (pick a model size preset)
2. Get a creative-writing dataset (TinyStories — purpose-built for small models learning to write coherent short stories)
3. Train a custom BPE tokenizer on that data
4. Pack the dataset into fast binary token files
5. Define the GPT architecture from scratch (attention, MLP, blocks)
6. Train with: mixed precision, grad accumulation, cosine LR schedule, gradient clipping
7. **Checkpointing**: autosave every N steps + on interrupt, auto-resume on restart, cross-session resume via Kaggle "Add Data → Notebook Output"
8. Generate sample stories
9. **Roadmap for Phase 2**: turning this base model into a "chattable" instruction-following model (SFT)

> 💡 **Before running:** In the Kaggle notebook settings (right sidebar), turn **Internet ON** and set the **Accelerator to GPU (T4 x2 is recommended)**. This notebook needs internet the first time to download the dataset.


## 1. Setup & Config

In [ ]:

# Install extra deps not preinstalled on Kaggle images (torch/numpy/tqdm already are)
!pip install -q datasets tokenizers==0.20.3


In [ ]:

import os, math, time, json, glob, random, signal, sys
from dataclasses import dataclass, asdict, field
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none")

# bf16 if supported (Ampere+), else fp16, else fp32
if DEVICE == "cuda" and torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability()[0] >= 8:
    PTDTYPE = torch.bfloat16
elif DEVICE == "cuda":
    PTDTYPE = torch.float16
else:
    PTDTYPE = torch.float32
print("Training dtype:", PTDTYPE)

autocast_ctx = (
    torch.amp.autocast(device_type="cuda", dtype=PTDTYPE)
    if DEVICE == "cuda" else nullcontext()
)


In [ ]:

# ---------------------------------------------------------------------------
# Model size presets. Start with "small" -- good story quality on TinyStories
# within a few hours on a Kaggle T4/P100. Bump to "medium" once the pipeline
# is verified to work end-to-end.
# ---------------------------------------------------------------------------
PRESETS = {
    "tiny":   dict(n_layer=4,  n_head=4,  n_embd=256, block_size=256),
    "small":  dict(n_layer=8,  n_head=8,  n_embd=512, block_size=512),
    "medium": dict(n_layer=12, n_head=12, n_embd=768, block_size=512),
}

PRESET_NAME = "small"     # <-- change this to "tiny" for a quick smoke test, or "medium" to scale up

@dataclass
class Config:
    # data
    dataset_name: str = "roneneldan/TinyStories"
    vocab_size: int = 8192          # trained fresh below, so this is a target, not fixed
    # model (filled from preset)
    n_layer: int = 8
    n_head: int = 8
    n_embd: int = 512
    block_size: int = 512
    dropout: float = 0.1
    bias: bool = False
    # training
    batch_size: int = 32            # per-step micro batch size
    grad_accum_steps: int = 4       # effective batch = batch_size * grad_accum_steps
    max_steps: int = 20000
    warmup_steps: int = 300
    lr: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_interval: int = 250
    eval_iters: int = 50
    save_interval: int = 500        # checkpoint every N optimizer steps
    sample_interval: int = 500      # generate a sample story every N steps
    keep_last_n_ckpts: int = 3      # rolling checkpoint retention (disk safety on Kaggle)
    # paths
    work_dir: str = "/kaggle/working"

cfg = Config(**PRESETS[PRESET_NAME])
print(cfg)

CKPT_DIR   = os.path.join(cfg.work_dir, "checkpoints")
DATA_DIR   = os.path.join(cfg.work_dir, "data")
TOK_DIR    = os.path.join(cfg.work_dir, "tokenizer")
SAMPLE_DIR = os.path.join(cfg.work_dir, "samples")
for d in [CKPT_DIR, DATA_DIR, TOK_DIR, SAMPLE_DIR]:
    os.makedirs(d, exist_ok=True)



## 2. Dataset — TinyStories

[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) is a synthetic dataset of short, simple stories written to only use words a 3-4 year old understands. It was specifically created (by Eldan & Li, Microsoft Research) to let **very small** language models learn to produce fluent, coherent, creative short stories — exactly our use case.

If you'd rather train on your own story corpus, replace the loading cell below with your own text — just make sure you end up with a `train_text` and `val_text` string (or iterable of strings).


In [ ]:

from datasets import load_dataset

RAW_TRAIN_TXT = os.path.join(DATA_DIR, "train.txt")
RAW_VAL_TXT   = os.path.join(DATA_DIR, "val.txt")

if not (os.path.exists(RAW_TRAIN_TXT) and os.path.exists(RAW_VAL_TXT)):
    print("Downloading TinyStories...")
    ds = load_dataset(cfg.dataset_name)
    train_split = ds["train"]
    val_split = ds["validation"] if "validation" in ds else ds["train"].select(range(2000))

    # A story-separator token so the tokenizer / model can learn document boundaries
    SEP = "<|endofstory|>"

    with open(RAW_TRAIN_TXT, "w", encoding="utf-8") as f:
        for row in train_split:
            f.write(row["text"].strip() + f"\n{SEP}\n")

    with open(RAW_VAL_TXT, "w", encoding="utf-8") as f:
        for row in val_split:
            f.write(row["text"].strip() + f"\n{SEP}\n")

    print("Saved raw text files.")
else:
    print("Raw text files already exist, skipping download.")

print("train.txt size (MB):", os.path.getsize(RAW_TRAIN_TXT) / 1e6)
print("val.txt size (MB):", os.path.getsize(RAW_VAL_TXT) / 1e6)



## 3. Train a Custom BPE Tokenizer

We train our **own** Byte-Pair Encoding tokenizer on the story corpus (rather than reusing GPT-2's) so the vocabulary is small and tuned to this simple children's-story vocabulary — this keeps the embedding table (and hence the model) small, which matters a lot at this scale.


In [ ]:

from tokenizers import ByteLevelBPETokenizer

TOK_JSON = os.path.join(TOK_DIR, "tokenizer.json")
SPECIAL_TOKENS = ["<|pad|>", "<|endofstory|>", "<|unk|>"]

if not os.path.exists(TOK_JSON):
    print("Training tokenizer...")
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train(
        files=[RAW_TRAIN_TXT],
        vocab_size=cfg.vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
    )
    tokenizer.save(TOK_JSON)
    print("Tokenizer trained and saved to", TOK_JSON)
else:
    print("Tokenizer already exists, loading it.")

from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(TOK_JSON)
VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id("<|pad|>")
EOS_ID = tokenizer.token_to_id("<|endofstory|>")
print("Vocab size:", VOCAB_SIZE, "| PAD:", PAD_ID, "| EOS:", EOS_ID)
cfg.vocab_size = VOCAB_SIZE



## 4. Pack Tokens Into Binary Files

Rather than tokenizing on the fly (slow, and re-done every epoch), we tokenize **once** and dump the whole corpus as a flat `uint16` array on disk. Training then just memory-maps this file and slices random windows out of it — this is fast and uses almost no RAM regardless of dataset size.


In [ ]:

TRAIN_BIN = os.path.join(DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(DATA_DIR, "val.bin")

def tokenize_to_bin(text_path, bin_path, chunk_chars=5_000_000):
    if os.path.exists(bin_path):
        print(f"{bin_path} already exists, skipping.")
        return
    print(f"Tokenizing {text_path} -> {bin_path}")
    ids_list = []
    with open(text_path, "r", encoding="utf-8") as f:
        buf = f.read()
    # chunk to keep tokenizer memory bounded on very large files
    for i in range(0, len(buf), chunk_chars):
        chunk = buf[i:i+chunk_chars]
        enc = tokenizer.encode(chunk)
        ids_list.append(np.array(enc.ids, dtype=np.uint16))
    ids = np.concatenate(ids_list)
    ids.tofile(bin_path)
    print(f"  -> {len(ids):,} tokens")

tokenize_to_bin(RAW_TRAIN_TXT, TRAIN_BIN)
tokenize_to_bin(RAW_VAL_TXT, VAL_BIN)

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")
print("Train tokens:", len(train_data), "| Val tokens:", len(val_data))


In [ ]:

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = np.random.randint(0, len(data) - cfg.block_size - 1, size=cfg.batch_size)
    x = np.stack([data[i:i+cfg.block_size].astype(np.int64) for i in ix])
    y = np.stack([data[i+1:i+1+cfg.block_size].astype(np.int64) for i in ix])
    x = torch.from_numpy(x)
    y = torch.from_numpy(y)
    if DEVICE == "cuda":
        x = x.pin_memory().to(DEVICE, non_blocking=True)
        y = y.pin_memory().to(DEVICE, non_blocking=True)
    else:
        x, y = x.to(DEVICE), y.to(DEVICE)
    return x, y



## 5. The Model — GPT-style Decoder-Only Transformer (from scratch)

Standard pre-norm Transformer decoder: causal self-attention + MLP blocks, learned positional embeddings, weight-tied input/output embeddings. Nothing exotic — this architecture (same family as GPT-2 / nanoGPT) is well understood, easy to train stably at small scale, and is what TinyStories-scale models are typically built with.


In [ ]:

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight  # weight tying

        self.apply(self._init_weights)
        # scaled init for residual projections (GPT-2 recipe)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than block_size"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos)[None, :, :])
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        n -= self.pos_emb.weight.numel()  # exclude pos emb, standard convention
        return n

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, top_p=0.95, eos_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            if top_p is not None:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cum_probs = torch.cumsum(sorted_probs, dim=-1)
                mask = cum_probs - sorted_probs > top_p
                sorted_probs[mask] = 0.0
                sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
                next_sorted = torch.multinomial(sorted_probs, 1)
                next_id = sorted_idx.gather(-1, next_sorted)
            else:
                next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
            if eos_id is not None and (next_id == eos_id).all():
                break
        self.train()
        return idx


In [ ]:

model = GPT(cfg).to(DEVICE)
print(f"Model params: {model.num_params()/1e6:.2f}M")
if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 7:
    try:
        model = torch.compile(model)
        print("torch.compile enabled")
    except Exception as e:
        print("torch.compile not used:", e)
elif DEVICE == "cuda":
    print("torch.compile disabled (requires CUDA capability >= 7.0)")



## 6. Optimizer & LR Schedule


In [ ]:

def configure_optimizer(model, weight_decay, lr):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2:
            decay.append(p)
        else:
            no_decay.append(p)
    groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95), eps=1e-8)

optimizer = configure_optimizer(model, cfg.weight_decay, cfg.lr)
scaler = torch.amp.GradScaler(enabled=(PTDTYPE == torch.float16))

def get_lr(step):
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / cfg.warmup_steps
    if step >= cfg.max_steps:
        return cfg.min_lr
    decay_ratio = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg.min_lr + coeff * (cfg.lr - cfg.min_lr)



## 7. Checkpointing (the important part!)

Kaggle sessions time out, disconnect, or you may just want to stop and resume later. This section makes training **fully resumable**:

- **`save_checkpoint`** dumps model weights, optimizer state, scaler state, step count, and config into one `.pt` file, plus always overwrites a `latest.pt` pointer.
- **`find_latest_checkpoint`** searches *both* `/kaggle/working/checkpoints` (this session) *and* any extra read-only dirs you pass in (previous sessions' outputs added as input datasets) and returns whichever checkpoint has the highest step count.
- Training auto-resumes from that checkpoint if one is found — otherwise starts fresh.
- A `finally` block guarantees a checkpoint is saved even if you interrupt the kernel (Kaggle "Stop" button raises `KeyboardInterrupt`).

### How to resume training across Kaggle sessions
1. Run this notebook, let it train for a while, then **Save Version** (this commits `/kaggle/working` as the notebook's output).
2. Next session: open a **new version** of the notebook, and under "Add Data" attach the **previous version's output** as an input dataset.
3. Add its checkpoints folder path to `EXTRA_CKPT_SEARCH_DIRS` below (e.g. `/kaggle/input/<your-notebook-slug>/checkpoints`).
4. Re-run — it will detect the latest checkpoint and resume from that exact step.


In [ ]:

EXTRA_CKPT_SEARCH_DIRS = [
    # "/kaggle/input/story-slm-v1/checkpoints",   # <- example: add previous session's output dir here
]

def _extract_step(path):
    base = os.path.basename(path)
    try:
        return int(base.replace("ckpt_step", "").replace(".pt", ""))
    except ValueError:
        return -1

def find_latest_checkpoint():
    search_dirs = [CKPT_DIR] + EXTRA_CKPT_SEARCH_DIRS
    candidates = []
    for d in search_dirs:
        candidates += glob.glob(os.path.join(d, "ckpt_step*.pt"))
    if not candidates:
        return None
    return max(candidates, key=_extract_step)

def save_checkpoint(step, best_val_loss):
    path = os.path.join(CKPT_DIR, f"ckpt_step{step}.pt")
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model  # unwrap torch.compile
    torch.save({
        "step": step,
        "model_state": raw_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "config": asdict(cfg),
    }, path)
    latest_path = os.path.join(CKPT_DIR, "latest.pt")
    torch.save({"points_to": path}, latest_path)
    print(f"[checkpoint] saved step {step} -> {path}")

    # rolling retention: keep only the last N milestone checkpoints (disk safety)
    all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_step*.pt")), key=_extract_step)
    for old in all_ckpts[:-cfg.keep_last_n_ckpts]:
        os.remove(old)

def load_checkpoint(path):
    print(f"[checkpoint] loading {path}")
    ckpt = torch.load(path, map_location=DEVICE)
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    raw_model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    return ckpt["step"], ckpt["best_val_loss"]

start_step, best_val_loss = 0, float("inf")
resume_path = find_latest_checkpoint()
if resume_path is not None:
    start_step, best_val_loss = load_checkpoint(resume_path)
    start_step += 1
    print(f"Resuming from step {start_step}")
else:
    print("No checkpoint found -- starting fresh training run.")


## 8. Loss Estimation & Sample Generation Helpers

In [ ]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(cfg.eval_iters)
        for k in range(cfg.eval_iters):
            x, y = get_batch(split)
            with autocast_ctx:
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def sample_story(prompt="Once upon a time,", max_new_tokens=200, step=None):
    model.eval()
    ids = tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with autocast_ctx:
        out = model.generate(x, max_new_tokens=max_new_tokens, eos_id=EOS_ID)
    text = tokenizer.decode(out[0].tolist())
    model.train()
    if step is not None:
        with open(os.path.join(SAMPLE_DIR, f"sample_step{step}.txt"), "w", encoding="utf-8") as f:
            f.write(text)
    return text



## 9. Training Loop

Runs from `start_step` to `cfg.max_steps`, with gradient accumulation, gradient clipping, periodic validation, periodic checkpointing, and periodic sample story generation so you can watch quality improve in real time.


In [ ]:

def train():
    global best_val_loss
    step = start_step
    t0 = time.time()
    try:
        while step < cfg.max_steps:
            lr = get_lr(step)
            for g in optimizer.param_groups:
                g["lr"] = lr

            optimizer.zero_grad(set_to_none=True)
            accum_loss = 0.0
            for micro_step in range(cfg.grad_accum_steps):
                x, y = get_batch("train")
                with autocast_ctx:
                    _, loss = model(x, y)
                    loss = loss / cfg.grad_accum_steps
                scaler.scale(loss).backward()
                accum_loss += loss.item()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()

            if step % 10 == 0:
                dt = time.time() - t0
                print(f"step {step:6d} | loss {accum_loss:.4f} | lr {lr:.2e} | {dt:.1f}s")
                t0 = time.time()

            if step % cfg.eval_interval == 0 and step > 0:
                losses = estimate_loss()
                print(f"  [eval] step {step} | train {losses['train']:.4f} | val {losses['val']:.4f}")
                if losses["val"] < best_val_loss:
                    best_val_loss = losses["val"]

            if step % cfg.sample_interval == 0 and step > 0:
                story = sample_story(step=step)
                print(f"  [sample @ step {step}]\n{story[:400]}\n---")

            if step % cfg.save_interval == 0 and step > 0:
                save_checkpoint(step, best_val_loss)

            step += 1
    except KeyboardInterrupt:
        print("Interrupted -- saving checkpoint before exiting.")
    finally:
        save_checkpoint(step - 1 if step > start_step else step, best_val_loss)
        print("Final checkpoint saved. You can re-run this notebook (or a new version) to resume.")

train()


## 10. Generate Some Stories With the Trained Model

In [ ]:

prompts = [
    "Once upon a time,",
    "Lily and her dog went to the park and",
    "The little robot woke up in a garden and",
]
for p in prompts:
    print("PROMPT:", p)
    print(sample_story(prompt=p, max_new_tokens=180))
    print("=" * 60)



## 11. Save Final Artifacts

Everything you need to reuse this model later (for chat fine-tuning, or just more inference) lives in `/kaggle/working`:
- `checkpoints/` — model + optimizer state
- `tokenizer/tokenizer.json` — the trained tokenizer
- `samples/` — generated stories over time

When you **Save Version** on this notebook, all of this becomes downloadable / attachable as an input dataset to future notebooks.


In [ ]:

final_info = {
    "preset": PRESET_NAME,
    "config": asdict(cfg),
    "params_millions": round(model.num_params() / 1e6, 2),
    "vocab_size": VOCAB_SIZE,
}
with open(os.path.join(cfg.work_dir, "run_info.json"), "w") as f:
    json.dump(final_info, f, indent=2)
print(json.dumps(final_info, indent=2))
print("\nSaved everything under:", cfg.work_dir)



## 12. Roadmap: Making It "Chattable" (Phase 2)

This notebook only does **pretraining** — the model learns to continue text/write stories, but has no notion of "user"/"assistant" turns or instruction-following. To turn it into a chat model, in a **follow-up notebook** you would:

1. **Load this checkpoint** (`checkpoints/latest.pt`) and the same `tokenizer.json` — don't retrain the tokenizer, keep vocab identical.
2. **Add chat special tokens** to the tokenizer (e.g. `<|user|>`, `<|assistant|>`, `<|system|>`, `<|endofturn|>`) and resize the model's embedding + head layers to the new vocab size (copy over old rows, randomly init the new ones).
3. **Build an SFT (supervised fine-tuning) dataset**: pairs like
   ```
   <|user|> Write me a short story about a brave mouse. <|assistant|> Once upon a time... <|endofturn|>
   ```
   You can synthesize this cheaply by taking existing TinyStories and auto-generating a "prompt" for each story (e.g. with a rule-based template, or by asking a larger model to write a one-line instruction that the story satisfies).
4. **Fine-tune at a lower learning rate** (e.g. 1/10th of pretraining LR) for a small number of epochs, **masking the loss on the prompt/user tokens** so the model only learns to predict the assistant's reply.
5. Optionally follow with a lightweight preference-tuning step (DPO) if you want the chat responses to feel more natural/on-style.
6. Reuse the **exact same checkpointing pattern** from this notebook (it's fully generic) for the SFT run.

This notebook's `GPT` class, tokenizer, and checkpoint format are all designed to be reused as-is in that follow-up — you shouldn't need to rewrite anything, just load, extend the vocab, and keep training.
